# Stock-pledge-default-prediction
### Portfolio version of a team academic project

* A classification analysis for predicting stock pledge financing defaults using financial variables, baseline machine learning models (Logistic Regression and Decision Tree), Artificial Neural Networks (ANN with SMOTE), and Ensemble Learning (Random Forest, XGBoost, and AdaBoost).

* A classification Problem - small Data with Class imbalance
* Train Data: 1442 rows and 62 columns

In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
#import libraries:


import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

# import shap

# from ydata_profiling import ProfileReport 


from sklearn.model_selection import train_test_split

from sklearn.preprocessing import LabelEncoder

# import xgboost as xgb

# from imblearn.over_sampling import SMOTE
# from sklearn.model_selection import StratifiedKFold, GridSearchCV, RandomizedSearchCV
# from sklearn.model_selection import KFold, cross_val_score


# from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, make_scorer
# import optuna

import warnings
warnings.simplefilter("ignore")
pd.options.mode.chained_assignment = None  

pd.set_option('display.max_columns', None)

# Data wrangling

### Reading Train and Test (inferencing) Data

In [ ]:
# Load the training dataset from the local folder
train = pd.read_csv("train.csv")

print("Dataset shape:", train.shape)
train.head()

## Target Leakage Removal

`ST`, `audit opinion`, and `Downgrade or negative` were removed because they are too closely related to a company’s distress status and may cause target leakage.

The other variables were kept because they still provide useful financial and operational information for predicting default.

In [ ]:
# Remove accidental whitespace from column names
train.columns = train.columns.str.strip()

# Remove variables that contain post-event distress information
leakage_cols = [
    "ST",
    "audit opinion",
    "Downgrade or negative"
]

train = train.drop(columns=leakage_cols, errors="ignore")

print("Removed leakage variables:", leakage_cols)
print("Dataset shape after removal:", train.shape)

In [ ]:
# Data Type

# Select only categorical columns
categorical_columns = train.select_dtypes(include=['object', 'category']).columns

# Print categorical columns
print(categorical_columns)


In [ ]:
print("Dataset shape:", train.shape)

print("\nColumn names:")
print(train.columns)

print("\nData types:")
print(train.dtypes)

print("\nBasic statistics:")
display(train.describe())

In [ ]:
train.info()

In [ ]:
# Check missing values
missing = train.isnull().sum()

missing[missing > 0]

In [ ]:
train.duplicated().sum()

In [ ]:
# Check the target variable distribution
train["IsDefault"].value_counts()

In [ ]:
train["IsDefault"].value_counts(normalize=True) * 100

In [ ]:
# Plot the target variable distribution
sns.countplot(x="IsDefault", data=train)

plt.title("Target Variable Distribution")
plt.xlabel("IsDefault")
plt.ylabel("Count")

plt.show()

In [ ]:
# Check data types
train.dtypes.value_counts()

In [ ]:
train.select_dtypes(include="object").columns

In [ ]:
# Set stock code as index
train.set_index("Stock code", inplace=True)

train.head()

In [ ]:
# Check remaining object columns
train.select_dtypes(include="object").columns

In [ ]:
train["P/E ratio"].head(10)

In [ ]:
# Remove thousands separators and convert P/E ratio to numeric
train["P/E ratio"] = pd.to_numeric(
    train["P/E ratio"]
        .astype(str)
        .str.replace(",", "", regex=False),
    errors="coerce"
)

print("P/E ratio data type:", train["P/E ratio"].dtype)
print("Missing values after conversion:",
      train["P/E ratio"].isnull().sum())

In [ ]:
train["P/E ratio"].isnull().sum()

In [ ]:
# Check data types after preprocessing
train.dtypes.value_counts()

### Finding

Some values in `P/E ratio` contained commas, so the column was read as text. After removing the commas, it was converted to numeric successfully and no missing values remained.

In [ ]:
# Correlation analysis
plt.figure(figsize=(14, 10))

correlation_matrix = train.corr()

sns.heatmap(
    correlation_matrix,
    cmap="coolwarm",
    center=0
)

plt.title("Correlation Heatmap")
plt.show()

### Finding

Most of the variables are not strongly related to each other. However, a few pairs show high correlation, so they were checked further using VIF.

In [ ]:
# Distribution of selected numerical features
selected_features = [
    "Z-SCORE",
    "Stock Volatility",
    "P/E ratio",
    "ROA"
]

train[selected_features].hist(
    figsize=(12, 8),
    bins=30
)

plt.tight_layout()
plt.show()

### Finding

The four variables have quite different distributions. `P/E ratio` and `ROA` are more skewed, while the other variables are more concentrated.

In [ ]:
# Boxplots for selected numerical features

selected_features = [
    "Z-SCORE",
    "Stock Volatility",
    "P/E ratio",
    "ROA"
]

plt.figure(figsize=(12, 8))

for i, column in enumerate(selected_features):
    plt.subplot(2, 2, i + 1)
    sns.boxplot(y=train[column])
    plt.title(column)

plt.tight_layout()
plt.show()

### Finding

There are some obvious outliers, especially in `P/E ratio`. I kept them because they may be real values.

## Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Separate features and target
X = train.drop("IsDefault", axis=1)
y = train["IsDefault"]

# Split the data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("Training set:", X_train.shape)
print("Testing set:", X_test.shape)

### Finding

The split gave 1,153 rows for training and 289 for testing. I used stratify to keep the default proportion similar in both sets.

## Multicollinearity Assessment

Multicollinearity occurs when two or more predictor variables contain highly overlapping information. This may make model coefficients unstable and reduce interpretability.

To avoid using the testing data during feature selection, multicollinearity is assessed using the training set only. Highly correlated variables will first be identified, but no variables will be removed automatically.

In [ ]:
# Calculate absolute correlations using the training set only
corr_matrix_train = X_train.corr().abs()

# Keep only the upper triangle to avoid duplicate pairs
upper_triangle = corr_matrix_train.where(
    np.triu(np.ones(corr_matrix_train.shape), k=1).astype(bool)
)

# List feature pairs with very high correlation
high_corr_pairs = (
    upper_triangle.stack()
    .reset_index()
)

high_corr_pairs.columns = [
    "Feature 1",
    "Feature 2",
    "Absolute Correlation"
]

high_corr_pairs = high_corr_pairs[
    high_corr_pairs["Absolute Correlation"] >= 0.90
].sort_values(
    "Absolute Correlation",
    ascending=False
)

high_corr_pairs

In [ ]:
from statsmodels.stats.outliers_influence import variance_inflation_factor
from sklearn.preprocessing import StandardScaler

# Scale training predictors for numerical stability
X_train_vif = pd.DataFrame(
    StandardScaler().fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

# Calculate VIF for each predictor
vif_results = pd.DataFrame({
    "Feature": X_train_vif.columns,
    "VIF": [
        variance_inflation_factor(X_train_vif.values, i)
        for i in range(X_train_vif.shape[1])
    ]
}).sort_values("VIF", ascending=False)

vif_results.head(15)

In [ ]:
# Remove redundant predictors while keeping the most interpretable variable
multicollinear_drop = [
    "Asset liability ratio (excluding advance receipts)",
    "Asset liability ratio (total liabilities - contract liabilities - advance receipts)/(total assets - goodwill - contract liabilities - advance receipts)",
    "EBITDA interest cover",
    "Number of research institutions concerned"
]

# Keep the original feature sets unchanged and create reduced copies
X_train_reduced = X_train.drop(
    columns=multicollinear_drop,
    errors="ignore"
)

X_test_reduced = X_test.drop(
    columns=multicollinear_drop,
    errors="ignore"
)

print("Original number of features:", X_train.shape[1])
print("Reduced number of features:", X_train_reduced.shape[1])
print("Training and testing columns match:",
      X_train_reduced.columns.equals(X_test_reduced.columns))

In [ ]:
# Recalculate VIF after removing redundant predictors
X_train_reduced_vif = pd.DataFrame(
    StandardScaler().fit_transform(X_train_reduced),
    columns=X_train_reduced.columns,
    index=X_train_reduced.index
)

vif_results_reduced = pd.DataFrame({
    "Feature": X_train_reduced_vif.columns,
    "VIF": [
        variance_inflation_factor(X_train_reduced_vif.values, i)
        for i in range(X_train_reduced_vif.shape[1])
    ]
}).sort_values("VIF", ascending=False)

vif_results_reduced.head(15)

### Finding

A few variables were very similar, so four were removed.

I kept `Asset liability ratio` because it is easier to understand. I also kept `EBIT interest cover` and the log version of research reports because they were more suitable for the later models.

The highest VIF then dropped from about 89 to 6.77.

## Feature Standardization

In [ ]:
from sklearn.preprocessing import StandardScaler
import pandas as pd

# Standardise feature sets in-place overwriting X_train and X_test DataFrames
scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train), columns=X_train.columns, index=X_train.index)
X_test = pd.DataFrame(scaler.transform(X_test), columns=X_test.columns, index=X_test.index)

# Standardise reduced feature sets in-place
if 'X_train_reduced' in locals():
    scaler_reduced = StandardScaler()
    X_train_reduced = pd.DataFrame(scaler_reduced.fit_transform(X_train_reduced), columns=X_train_reduced.columns, index=X_train_reduced.index)
    X_test_reduced = pd.DataFrame(scaler_reduced.transform(X_test_reduced), columns=X_test_reduced.columns, index=X_test_reduced.index)

print("Standardised X_train shape:", X_train.shape)
print("Standardised X_test shape :", X_test.shape)

In [ ]:
# Verify standardisation results
standardisation_check = pd.DataFrame({
    "Dataset": ["Full features (X_train)", "Reduced features (X_train_reduced)"],
    "Mean": [
        X_train.values.mean(),
        X_train_reduced.values.mean() if 'X_train_reduced' in locals() else 0
    ],
    "Standard deviation": [
        X_train.values.std(),
        X_train_reduced.values.std() if 'X_train_reduced' in locals() else 1
    ]
})

standardisation_check

In [ ]:
# Final preprocessing validation
assert X_train.shape == (1153, 57)
assert X_test.shape == (289, 57)

assert X_train.columns.equals(X_test.columns)
assert X_train.isnull().sum().sum() == 0
assert X_test.isnull().sum().sum() == 0

assert len(y_train) == len(X_train)
assert len(y_test) == len(X_test)

print("Preprocessing validation passed.")
print("Full standardized features (X_train):", X_train.shape[1])

# Logistic Regression

Logistic Regression is selected as the baseline model because it is a simple and interpretable algorithm that is widely used for binary classification problems. A baseline model with default hyperparameters is first established to provide a reference for evaluating the effectiveness of subsequent hyperparameter tuning.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Build a baseline Logistic Regression model
lr = LogisticRegression(random_state=42, max_iter=1000)

# Train the model
lr.fit(X_train, y_train)

# Predict class labels
y_pred = lr.predict(X_test)

# Predict class probabilities for ROC-AUC
y_prob = lr.predict_proba(X_test)[:,1]

# Evaluate model performance
print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision:", precision_score(y_test, y_pred))
print("Recall:", recall_score(y_test, y_pred))
print("F1:", f1_score(y_test, y_pred))
print("ROC AUC:", roc_auc_score(y_test, y_prob))

print(confusion_matrix(y_test, y_pred))
print(classification_report(y_test, y_pred))

The baseline Logistic Regression model achieved an accuracy of 89.96% and an ROC-AUC score of 0.94, indicating good overall discrimination between default and non-default companies. However, the model achieved a recall of only 62.26% for the default class, meaning that a considerable number of default companies were incorrectly classified as non-default. Although the precision for default predictions reached 78.57%, the relatively low recall resulted in an F1-score of 0.69. Since identifying default companies is the primary objective of this study, further hyperparameter tuning was conducted to improve the model's ability to detect default cases.

## Optimized Logistic Regression
To improve the baseline model, GridSearchCV is used to search for the optimal combination of hyperparameters. The optimized model is then evaluated and compared with the baseline Logistic Regression model.

In [ ]:
from sklearn.model_selection import GridSearchCV

# Define the parameter grid
param_grid = {
    'C': [0.01, 0.1, 1, 10],
    'penalty': ['l2'],
    'solver': ['lbfgs']
}

# Grid Search
grid_lr = GridSearchCV(
    LogisticRegression(max_iter=1000, random_state=42),
    param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_lr.fit(X_train, y_train)

print("Best Parameters:", grid_lr.best_params_)
print("Best CV Score:", grid_lr.best_score_)

In [ ]:
best_lr = grid_lr.best_estimator_

y_pred_best = best_lr.predict(X_test)
y_prob_best = best_lr.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred_best))
print("Precision:", precision_score(y_test, y_pred_best))
print("Recall:", recall_score(y_test, y_pred_best))
print("F1:", f1_score(y_test, y_pred_best))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best))

print(confusion_matrix(y_test, y_pred_best))
print(classification_report(y_test, y_pred_best))

### Performance Comparison
GridSearchCV selected the same hyperparameter configuration as the baseline Logistic Regression model. As a result, the optimised model achieved identical performance on the test set, including accuracy, precision, recall, F1-score and ROC-AUC. This indicates that the baseline Logistic Regression model was already well configured for the dataset, and further hyperparameter tuning did not lead to additional performance improvement.

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

RocCurveDisplay.from_estimator(best_lr, X_test, y_test)
plt.title("ROC Curve - Optimized Logistic Regression")
plt.show()

# Decision Tree

## Baseline Decision Tree
Decision Tree is selected as the second classification model because it can capture nonlinear relationships and is easy to interpret. A baseline Decision Tree model is first established before hyperparameter tuning to evaluate its initial performance.

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)

# Build the baseline Decision Tree model
dt = DecisionTreeClassifier(random_state=42)

# Train the model
dt.fit(X_train, y_train)

# Make predictions
y_pred_dt = dt.predict(X_test)
y_prob_dt = dt.predict_proba(X_test)[:, 1]

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred_dt))
print("Precision:", precision_score(y_test, y_pred_dt))
print("Recall:", recall_score(y_test, y_pred_dt))
print("F1:", f1_score(y_test, y_pred_dt))
print("ROC AUC:", roc_auc_score(y_test, y_prob_dt))

print(confusion_matrix(y_test, y_pred_dt))
print(classification_report(y_test, y_pred_dt))

## Model Evaluation

The baseline Decision Tree model achieved an accuracy of 91.00% and an ROC-AUC score of 0.8571, demonstrating good overall classification performance. The model achieved a recall of 77.36% for the default class, indicating that it was able to identify a relatively high proportion of default companies. However, the precision was 74.55%, suggesting that some non-default companies were incorrectly classified as defaults. Overall, the baseline Decision Tree provided a good balance between detecting default companies and overall predictive performance. To further improve the model, GridSearchCV was applied to optimise the hyperparameters.

In [ ]:
from sklearn.model_selection import GridSearchCV

param_grid = {
    'criterion': ['gini', 'entropy'],
    'max_depth': [2, 3, 5],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4]
}

grid_dt = GridSearchCV(
    DecisionTreeClassifier(random_state=42),
    param_grid,
    cv=5,
    scoring='recall',
    n_jobs=-1
)

grid_dt.fit(X_train, y_train)

print("Best Parameters:", grid_dt.best_params_)
print("Best CV Score:", grid_dt.best_score_)

In [ ]:
# Evaluate optimized Decision Tree on both Training and Testing sets
best_dt = grid_dt.best_estimator_

# Training Set Predictions
y_train_pred_dt = best_dt.predict(X_train)
y_train_prob_dt = best_dt.predict_proba(X_train)[:, 1]

# Testing Set Predictions
y_test_pred_dt = best_dt.predict(X_test)
y_test_prob_dt = best_dt.predict_proba(X_test)[:, 1]

print("=== Decision Tree: TRAINING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_train, y_train_pred_dt))
print("Precision:", precision_score(y_train, y_train_pred_dt))
print("Recall   :", recall_score(y_train, y_train_pred_dt))
print("F1 Score :", f1_score(y_train, y_train_pred_dt))
print("ROC AUC  :", roc_auc_score(y_train, y_train_prob_dt))
print("\nTraining Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred_dt))
print("\nTraining Classification Report:")
print(classification_report(y_train, y_train_pred_dt))

print("\n" + "="*50 + "\n")

print("=== Decision Tree: TESTING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred_dt))
print("Precision:", precision_score(y_test, y_test_pred_dt))
print("Recall   :", recall_score(y_test, y_test_pred_dt))
print("F1 Score :", f1_score(y_test, y_test_pred_dt))
print("ROC AUC  :", roc_auc_score(y_test, y_test_prob_dt))
print("\nTesting Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_dt))
print("\nTesting Classification Report:")
print(classification_report(y_test, y_test_pred_dt))

## Hyperparameter Tuning

Compared with the baseline model, the optimised Decision Tree achieved improvements across all evaluation metrics. Accuracy increased from 91.00% to 96.89%, precision improved from 74.55% to 95.83%, recall increased from 77.36% to 86.79%, F1-score improved from 0.7593 to 0.9109, and ROC-AUC increased from 0.8571 to 0.9644. These results indicate that the optimised Decision Tree achieved stronger predictive performance and a better balance between identifying default companies and reducing false positive predictions.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_importance = pd.DataFrame({
    'Feature': X_train.columns,
    'Importance': best_dt.feature_importances_
})

feature_importance = feature_importance.sort_values(
    by='Importance',
    ascending=False
).head(10)

plt.figure(figsize=(8,6))
plt.barh(feature_importance['Feature'],
         feature_importance['Importance'])
plt.gca().invert_yaxis()

plt.title("Top 10 Feature Importances")
plt.xlabel("Importance")
plt.show()

The figure shows the ten most influential features used by the Decision Tree model. Features with higher importance contribute more to the model's prediction of stock pledge default.

In [ ]:
from sklearn.tree import plot_tree
import matplotlib.pyplot as plt

plt.figure(figsize=(20,10))
plot_tree(
    best_dt,
    filled=True,
    feature_names=X_train.columns,
    class_names=['Non-default', 'Default'],
    rounded=True,
    fontsize=8
)

plt.show()

## 1. Baseline & SMOTE-Enhanced MLPClassifier

# Artificial Neural Network (ANN) with SMOTE & Recall-Targeted GridSearchCV

## 1. Baseline & SMOTE-Enhanced MLPClassifier
Artificial Neural Network (ANN) using Multi-Layer Perceptron (`MLPClassifier`) is implemented. To handle class imbalance (18% default rate) and maximize default recall:
1. Features are standardized using `StandardScaler` (`X_train_scaled`, `X_test_scaled`).
2. Synthetic Minority Over-sampling Technique (`SMOTE`) is applied to balance `X_train_scaled`.
3. The `alpha` parameter controls L2 regularization to penalize overfitting.
4. Early Stopping (`early_stopping=True`, `n_iter_no_change=10`) is enabled to prevent neural network overfitting.

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt

# 1. Feature Standardization
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# 2. Apply SMOTE to balance minority default samples in scaled training set
smote = SMOTE(random_state=42)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train_scaled, y_train)

# 3. Build MLPClassifier with Early Stopping & (50, 25) Architecture
mlp = MLPClassifier(
    solver='adam',
    activation='logistic',
    hidden_layer_sizes=(100,),
    learning_rate_init=0.001,
    alpha=0.0001,
    max_iter=1000,
    early_stopping=True,
    validation_fraction=0.1,
    n_iter_no_change=10,
    random_state=42
)

# Train ANN model on SMOTE-resampled dataset
mlp.fit(X_train_resampled, y_train_resampled)


In [ ]:
# Evaluate baseline ANN on SMOTE-resampled Training set
y_train_pred_mlp = mlp.predict(X_train_resampled)
y_train_prob_mlp = mlp.predict_proba(X_train_resampled)[:, 1]

print("=== Baseline ANN: TRAINING SET (SMOTE) PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_train_resampled, y_train_pred_mlp))
print("Precision:", precision_score(y_train_resampled, y_train_pred_mlp))
print("Recall   :", recall_score(y_train_resampled, y_train_pred_mlp))
print("F1 Score :", f1_score(y_train_resampled, y_train_pred_mlp))
print("ROC AUC  :", roc_auc_score(y_train_resampled, y_train_prob_mlp))

print("\nClassification Report:")
print(classification_report(y_train_resampled, y_train_pred_mlp))


# Predict on scaled test set
y_pred_mlp = mlp.predict(X_test_scaled)
y_prob_mlp = mlp.predict_proba(X_test_scaled)[:, 1]

# Evaluate SMOTE-Enhanced ANN performance
print("=== SMOTE-Enhanced Baseline ANN Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_mlp))
print("Precision:", precision_score(y_test, y_pred_mlp))
print("Recall:", recall_score(y_test, y_pred_mlp))
print("F1 Score:", f1_score(y_test, y_pred_mlp))
print("ROC AUC:", roc_auc_score(y_test, y_prob_mlp))

print("\nClassification Report:")
print(classification_report(y_test, y_pred_mlp))

**Model Performance:**

Among the 289 test samples, the model achieved a recall of 87%, successfully identifying the majority of defaulting companies. It also obtained an F1-score of 0.72, indicating a practical balance between precision and recall for the Techstars credit risk assessment scenario. 

However, the model exhibited a relatively high false positive rate, with a precision of only 0.62, meaning that approximately 40% of the companies predicted as defaults were actually non-defaulting.

This may lead to unnecessary auditing and investigation costs for the company. Therefore, although the initial ANN model demonstrates good performance in identifying most defaulting companies, there remains room for further optimization to reduce false positives while maintaining high recall.

In [ ]:
# Plot ANN Training Loss Curve
plt.figure(figsize=(8, 5))
plt.plot(mlp.loss_curve_, color='blue', linewidth=2)
plt.title("ANN Training Loss Curve (with Early Stopping)")
plt.xlabel("Iterations (Epochs)")
plt.ylabel("Loss")
plt.grid(True)
plt.show()

## 2. Hyperparameter Tuning for ANN with SMOTE & Recall-Targeted GridSearchCV

GridSearchCV with **5-fold cross-validation** is used to find optimal hyperparameters:

In financial default prediction, missing a default company carries extreme financial risk. Therefore, `GridSearchCV` uses `scoring='recall'` to specifically optimize hyperparameter combinations (`hidden_layer_sizes`, `activation`, `learning_rate_init`,`alpha`) that maximize default detection (Recall).

In [ ]:
# from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import randint, uniform
from imblearn.pipeline import Pipeline as ImbPipeline

# Pipeline
pipeline = ImbPipeline([
    ('smote', SMOTE(random_state=42)),
    ('mlp', MLPClassifier(
        solver='adam',
        max_iter=1000,
        early_stopping=True,
        validation_fraction=0.1,
        n_iter_no_change=10,
        random_state=42,
        alpha=0.001
    ))
])

param_dist = {
    'mlp__hidden_layer_sizes': [
        (50, 25), (64, 32), (100, 50), (128, 64),
        (100,), (64,), (64, 32, 16), (100, 50, 25),
        (64, 64), (256, 128), (128, 64, 32),
    ],
    'mlp__activation': ['logistic', 'tanh'],
    'mlp__learning_rate_init': uniform(0.0005, 0.005),  # 0.0005~0.0055
    'mlp__alpha': uniform(0.0001, 0.1),                  # 0.0001~0.1001
}

random_mlp = RandomizedSearchCV(
    pipeline,
    param_dist,
    n_iter=30,           
    cv=5,
    scoring='recall',
    n_jobs=-1,
    random_state=42
)

random_mlp.fit(X_train_scaled, y_train)
print("Best Parameters:", random_mlp.best_params_)


In [ ]:
# Evaluate optimized ANN model from RandomizedSearchCV
best_mlp = random_mlp.best_estimator_  # ← grid_mlp → random_mlp

y_prob_best_mlp = best_mlp.predict_proba(X_test_scaled)[:, 1]

# Threshold Tuning: Optimizing the Precision-Recall Trade-off
y_pred_best_mlp = (y_prob_best_mlp > 0.49).astype(int)  

print("=== Recall-Optimized ANN Performance ===")
print("Accuracy:", accuracy_score(y_test, y_pred_best_mlp))
print("Precision:", precision_score(y_test, y_pred_best_mlp))
print("Recall:", recall_score(y_test, y_pred_best_mlp))
print("F1 Score:", f1_score(y_test, y_pred_best_mlp))
print("ROC AUC:", roc_auc_score(y_test, y_prob_best_mlp))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred_best_mlp))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_best_mlp))


In [ ]:
from sklearn.metrics import RocCurveDisplay

# Plot ROC Curve for Recall-Optimized ANN
RocCurveDisplay.from_estimator(best_mlp, X_test_scaled, y_test)
plt.title("ROC Curve - Recall-Optimized ANN")
plt.show()

## ANN Evaluation Summary
By setting `scoring='recall'` in `GridSearchCV`, the Artificial Neural Network achieves a high **Recall of 90.6%**, an Accuracy of 87.19%, F1-score of 0.7218, and an ROC-AUC of **0.9404** on restored feature sets.

Based on the above performance, the model achieves **satisfactory monitoring results.**(90.6%)

It is recommended that companies with a default probability above 50% be directly rejected or approached with caution, serving as a red line for risk control. For medium-risk clients with default probabilities between 40% and 50%, a manual review process should be triggered, where investment managers conduct secondary assessments based on qualitative factors such as management background and industry trends.

# Model Improvement

## 1. ElasticNet Regularized Logistic Regression
1.ElasticNet Logistic Regression applies both L1 and L2 regularization to avoid overfitting and and improve model interpretability.

2.In the model parameters adjustments, we still used 5-fold cross-validation (GridSearchCV) to find the best combination of parameters. We also tested different values of C and l1_ratio.

3.The target is still default detection. We used recall as the optimization measure to reduce the amount of missed defaults and to increase the model’s capacity to correctly identify defaulting enterprises.

4.Compared to AdaBoost or Random Forests, our model can further determine which financial factors raise or reduce the probability of default.


### Baseline ElasticNet Model

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    roc_auc_score, confusion_matrix, classification_report
)
import numpy as np
import pandas as pd

# Step 1: Baseline ElasticNet Model (Heavy Regularization Baseline: C=0.05, l1_ratio=0.9)
en_base = LogisticRegression(
    penalty='elasticnet', solver='saga', C=0.05, l1_ratio=0.9,
    random_state=42, max_iter=2000
)
en_base.fit(X_train, y_train)

y_test_pred_base = en_base.predict(X_test)
y_test_prob_base = en_base.predict_proba(X_test)[:, 1]

print("=== Baseline ElasticNet Model: TESTING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred_base))
print("Precision:", precision_score(y_test, y_test_pred_base))
print("Recall   :", recall_score(y_test, y_test_pred_base))
print("F1 Score :", f1_score(y_test, y_test_pred_base))
print("ROC AUC  :", roc_auc_score(y_test, y_test_prob_base))
print("\nTesting Confusion Matrix (Baseline):")
print(confusion_matrix(y_test, y_test_pred_base))

### Optimized ElasticNet Model

In [ ]:
from sklearn.model_selection import GridSearchCV

# Step 2: Optimized ElasticNet Model via 5-Fold GridSearchCV Dual Hyperparameter Tuning
param_grid_en = {
    'C': [0.01, 0.1, 0.5, 1, 5, 10],
    'l1_ratio': [0.1, 0.3, 0.5, 0.7, 0.9],
    'solver': ['saga'],
    'penalty': ['elasticnet']
}

grid_en = GridSearchCV(
    LogisticRegression(random_state=42, max_iter=2000),
    param_grid_en, cv=5, scoring='recall', n_jobs=-1
)
grid_en.fit(X_train, y_train)

best_en = grid_en.best_estimator_
print("=== Optimized ElasticNet GridSearchCV Dual Hyperparameter Optimization ===")
print("Best Parameters found:", grid_en.best_params_)
print("Best CV Recall Score  :", grid_en.best_score_)

y_train_pred_en = best_en.predict(X_train)
y_train_prob_en = best_en.predict_proba(X_train)[:, 1]

y_test_pred_en = best_en.predict(X_test)
y_test_prob_en = best_en.predict_proba(X_test)[:, 1]

print("\n=== Optimized ElasticNet Model: TRAINING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_train, y_train_pred_en))
print("Precision:", precision_score(y_train, y_train_pred_en))
print("Recall   :", recall_score(y_train, y_train_pred_en))
print("F1 Score :", f1_score(y_train, y_train_pred_en))
print("ROC AUC  :", roc_auc_score(y_train, y_train_prob_en))
print("\nTraining Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred_en))

print("\n" + "="*50 + "\n")

print("=== Optimized ElasticNet Model: TESTING SET PERFORMANCE (Post-Tuning) ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred_en))
print("Precision:", precision_score(y_test, y_test_pred_en))
print("Recall   :", recall_score(y_test, y_test_pred_en))
print("F1 Score :", f1_score(y_test, y_test_pred_en))
print("ROC AUC  :", roc_auc_score(y_test, y_test_prob_en))
print("\nTesting Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_en))
print("\nTesting Classification Report:")
print(classification_report(y_test, y_test_pred_en))

### Finding
All metrics showed that the improved ElasticNet model performed better than the baseline model. Furthermore, the recall increase from 50.9% to 60.4%, meaning the improved model was capable of detecting more default companies following the hyper-parameter adjustments.

In addition, the training and testing outcomes are compared to determine whether the model is over-fitting. Performance of the model is comparable on train and test sets and there are no evident symptoms of overfitting.

### Feature Coefficient Analysis

In [ ]:
# Extract and display top influential feature coefficients & odds ratios
coef_df = pd.DataFrame({
    'Feature': X_train.columns,
    'Coefficient (Beta)': best_en.coef_[0],
    'Odds Ratio (exp(Beta))': np.exp(best_en.coef_[0])
}).sort_values(by='Coefficient (Beta)', key=abs, ascending=False)

print("=== TOP 10 INFLUENTIAL FINANCIAL FEATURES & ODDS RATIOS ===")
print(coef_df.head(10).to_string(index=False))

# Plot Top 10 Feature Coefficients
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 6))
top10 = coef_df.head(10).sort_values(by='Coefficient (Beta)', ascending=True)
colors = ['#d9534f' if c > 0 else '#5cb85c' for c in top10['Coefficient (Beta)']]
plt.barh(top10['Feature'], top10['Coefficient (Beta)'], color=colors)
plt.xlabel("Regression Coefficient (Beta)")
plt.title("Top 10 Feature Coefficients - ElasticNet Regularized Model")
plt.grid(True, linestyle='--', alpha=0.6)
plt.show()

### Finding
Compared to decision trees, which only offer a ranking of the significance of features, the Optimised ElasticNet Model also indicate the direction of the effect of each variable on the probability of default. It also enables the amount of the influence of each variable to be assessed using regression coefficients and odds ratios. For example, a negative coefficient on the cash-to-income ratio implies that better cash generation capabilities is related with a reduced likelihood of default.

In [ ]:
from sklearn.metrics import RocCurveDisplay
import matplotlib.pyplot as plt

RocCurveDisplay.from_estimator(best_en, X_test, y_test)
plt.title("ROC Curve - ElasticNet Regularized Model - testing set")
plt.show()

## 2. Bagging Model: Random Forest Classifier with SMOTE
Applying SMOTE to the training data increases the representation of the minority default class, enabling the Random Forest classifier to better recognise companies with potential default risk. Random Forest is selected as the bagging model because it constructs multiple decision trees using different bootstrap samples and combines their predictions through majority voting, resulting in more stable predictions and reduced overfitting compared with a single decision tree.

In this model, 100 decision trees (`n_estimators = 100`) are constructed to balance predictive performance and model complexity. The model is then evaluated on both the training and testing datasets to assess its predictive performance and determine whether it generalises well to unseen data.

In [ ]:
from sklearn.ensemble import RandomForestClassifier
from imblearn.over_sampling import SMOTE
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)
import matplotlib.pyplot as plt

# Apply SMOTE for Random Forest
smote = SMOTE(random_state=42)
X_train_rf_smote, y_train_rf_smote = smote.fit_resample(X_train, y_train)

# Build SMOTE-Enhanced Random Forest Classifier
rf_smote = RandomForestClassifier(n_estimators=100, random_state=42)
rf_smote.fit(X_train_rf_smote, y_train_rf_smote)

# Evaluate Random Forest on Training set
y_train_pred_rf = rf_smote.predict(X_train)
y_train_prob_rf = rf_smote.predict_proba(X_train)[:, 1]

# Evaluate Random Forest on Testing set
y_test_pred_rf = rf_smote.predict(X_test)
y_test_prob_rf = rf_smote.predict_proba(X_test)[:, 1]

print("=== Random Forest (Bagging): TRAINING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_train, y_train_pred_rf))
print("Precision:", precision_score(y_train, y_train_pred_rf))
print("Recall   :", recall_score(y_train, y_train_pred_rf))
print("F1 Score :", f1_score(y_train, y_train_pred_rf))
print("ROC AUC  :", roc_auc_score(y_train, y_train_prob_rf))
print("\nTraining Confusion Matrix:")
print(confusion_matrix(y_train, y_train_pred_rf))

print("\n" + "="*50 + "\n")

print("=== Random Forest (Bagging): TESTING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred_rf))
print("Precision:", precision_score(y_test, y_test_pred_rf))
print("Recall   :", recall_score(y_test, y_test_pred_rf))
print("F1 Score :", f1_score(y_test, y_test_pred_rf))
print("ROC AUC  :", roc_auc_score(y_test, y_test_prob_rf))
print("\nTesting Confusion Matrix:")
print(confusion_matrix(y_test, y_test_pred_rf))
print("\nTesting Classification Report:")
print(classification_report(y_test, y_test_pred_rf))

The Random Forest model achieved excellent performance on both the training and testing datasets. While all evaluation metrics reached 1.00 on the training set, suggesting that the model fitted the training data extremely well, the testing results remained strong, with an accuracy of 92.0% and a ROC-AUC score of 0.964. The confusion matrix and classification report further indicate that the model effectively distinguished between default and non-default companies while maintaining a good balance between precision and recall for the minority default class. These results suggest that the model has strong predictive capability and can provide valuable support for identifying potential default companies. Overall, the Random Forest model provides a strong starting point for the subsequent boosting models, making it easier to evaluate whether more advanced ensemble methods can further improve predictive performance.

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(rf_smote, X_test, y_test)
plt.title("ROC Curve - Random Forest Classifier - Testing Set")
plt.show()

## 3. Adaptive Boosting Model: AdaBoost Classifier (Comparison Model)
In the early stage of model building, we first tested the performance of the AdaBoost classifier and kept the experimental data for AdaBoost for comparative reasons.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier

ada_base = AdaBoostClassifier(
    estimator=DecisionTreeClassifier(max_depth=1, random_state=42),
    n_estimators=50,
    learning_rate=1.0,
    random_state=42
)
ada_base.fit(X_train, y_train)

### AdaBoost on Training set
y_train_pred_ada = ada_base.predict(X_train)
y_train_prob_ada = ada_base.predict_proba(X_train)[:, 1]

### AdaBoost on Testing set
y_test_pred_ada = ada_base.predict(X_test)
y_test_prob_ada = ada_base.predict_proba(X_test)[:, 1]

print("=== Baseline AdaBoost: TRAINING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_train, y_train_pred_ada))
print("Precision:", precision_score(y_train, y_train_pred_ada))
print("Recall   :", recall_score(y_train, y_train_pred_ada))
print("F1 Score :", f1_score(y_train, y_train_pred_ada))
print("ROC AUC  :", roc_auc_score(y_train, y_train_prob_ada))

print("\n" + "="*50 + "\n")

print("=== Baseline AdaBoost: TESTING SET PERFORMANCE ===")
print("Accuracy :", accuracy_score(y_test, y_test_pred_ada))
print("Precision:", precision_score(y_test, y_test_pred_ada))
print("Recall   :", recall_score(y_test, y_test_pred_ada))
print("F1 Score :", f1_score(y_test, y_test_pred_ada))
print("ROC AUC  :", roc_auc_score(y_test, y_test_prob_ada))

In [ ]:
from sklearn.metrics import RocCurveDisplay

RocCurveDisplay.from_estimator(ada_base, X_test, y_test)
plt.title("ROC Curve - AdaBoost Classifier - testing set")
plt.show()